In [1]:
# Carrega os dados já limpos e prontos de acordo com a análise exploratoria para serem usados

from src.preparo import carregar_particao

X_treino, X_teste, y_treino, y_teste = carregar_particao()

In [ ]:
# Monta o modelo: escalonamento + rede neural em um único Pipeline (ainda sem treinar)

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.preparo import SEED

modelo = Pipeline([
    ('escalonador', StandardScaler()),
    ('rede', MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        alpha=0.00031489116479568613,
        learning_rate_init=0.00023273922280628724,
        early_stopping=True,
        random_state=SEED,
    )),
])


In [3]:
# Treina o pipeline: o escalonador é ajustado no treino e depois a rede aprende com os dados escalonados

modelo.fit(X_treino, y_treino)

rede = modelo['rede']
print(f'Épocas executadas: {rede.n_iter_}')
print(f'Melhor acurácia na validação interna: {rede.best_validation_score_:.5f}')


Épocas executadas: 14
Melhor acurácia na validação interna: 0.99982


In [4]:
# Previsões out-of-fold no treino: cada transação é prevista por um modelo que não a viu no treino

from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import cross_val_predict

from src.preparo import cv_tuning

prob_oof = cross_val_predict(modelo, X_treino, y_treino, cv=cv_tuning(), method='predict_proba')[:, 1]

print(f'Average Precision (out-of-fold): {average_precision_score(y_treino, prob_oof):.4f}')
print(f'ROC-AUC (out-of-fold): {roc_auc_score(y_treino, prob_oof):.4f}')


Average Precision (out-of-fold): 0.8311
ROC-AUC (out-of-fold): 0.9727


In [5]:
# Escolhe o limiar que maximiza o F2 nas previsões out-of-fold (nunca 0,5 fixo)

import numpy as np
from sklearn.metrics import fbeta_score, precision_recall_curve

precisao, recall, limiares = precision_recall_curve(y_treino, prob_oof)

# precisao e recall têm um ponto a mais que limiares (o ponto final, sem limiar); descartamos esse
precisao, recall = precisao[:-1], recall[:-1]

# F2 = 5·P·R / (4·P + R); onde o denominador é zero, o F2 vale 0
denominador = 4 * precisao + recall
f2 = np.divide(5 * precisao * recall, denominador, out=np.zeros_like(denominador), where=denominador > 0)

melhor = np.argmax(f2)
limiar = limiares[melhor]

print(f'Limiar escolhido: {limiar:.4f}')
print(f'F2: {f2[melhor]:.4f} | precisão: {precisao[melhor]:.4f} | recall: {recall[melhor]:.4f}')
print(f'F2 com limiar 0,5: {fbeta_score(y_treino, prob_oof >= 0.5, beta=2):.4f}')


Limiar escolhido: 0.3153
F2: 0.8271 | precisão: 0.8800 | recall: 0.8148
F2 com limiar 0,5: 0.7929


In [ ]:
# Ajusta os hiperparâmetros da rede com busca aleatória (métrica: Average Precision nos folds do treino)

from joblib import parallel_backend
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

from src.preparo import cv_tuning

espaco = {
    'rede__hidden_layer_sizes': [(32,), (64,), (64, 32), (128, 64), (128, 64, 32)],
    'rede__alpha': loguniform(1e-5, 1e-1),
    'rede__learning_rate_init': loguniform(1e-4, 1e-2),
}

busca = RandomizedSearchCV(modelo, espaco, n_iter=20, scoring='average_precision', cv=cv_tuning(), random_state=SEED, n_jobs=5)

# Backend de threads: com processos (o padrão do joblib) o Python caiu com falha de segmentação neste ambiente
with parallel_backend('threading', n_jobs=5):
    busca.fit(X_treino, y_treino)

print(busca.best_params_)
print(f'Melhor AP médio nos folds: {busca.best_score_:.4f}')



{'rede__alpha': np.float64(0.00031489116479568613), 'rede__hidden_layer_sizes': (128, 64, 32), 'rede__learning_rate_init': np.float64(0.00023273922280628724)}
Melhor AP médio nos folds: 0.8430


In [7]:
# Avaliação final no teste (usado uma única vez): probabilidades do modelo ajustado + limiar escolhido no treino

from sklearn.metrics import (average_precision_score, confusion_matrix, fbeta_score,
                             precision_score, recall_score, roc_auc_score)

prob_teste = modelo.predict_proba(X_teste)[:, 1]
previsao = prob_teste >= limiar

print(f'Average Precision: {average_precision_score(y_teste, prob_teste):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_teste, prob_teste):.4f}')
print(f'F2: {fbeta_score(y_teste, previsao, beta=2):.4f}')
print(f'Precisão: {precision_score(y_teste, previsao):.4f} | Recall: {recall_score(y_teste, previsao):.4f}')

(vn, fp), (fn, vp) = confusion_matrix(y_teste, previsao)
print(f'Verdadeiros negativos: {vn} | Falsos positivos: {fp} | Falsos negativos: {fn} | Verdadeiros positivos: {vp}')


Average Precision: 0.8250
ROC-AUC: 0.9566
F2: 0.7843
Precisão: 0.9114 | Recall: 0.7579
Verdadeiros negativos: 56644 | Falsos positivos: 7 | Falsos negativos: 23 | Verdadeiros positivos: 72


In [ ]:
# Essas foram as métricas obtidas pelo último RUN do algortimo:

"""Average Precision: 0.8250
ROC-AUC: 0.9566
F2: 0.7843
Precisão: 0.9114 | Recall: 0.7579
Verdadeiros negativos: 56644 | Falsos positivos: 7 | Falsos negativos: 23 | Verdadeiros positivos: 72"""

# Diante dessas métricas podemos inferir que:
# A rede neural de três camadas ocultas (128,64,32), parâmetro alpha=0.00031489116479568613 e learning rate de 0.00023273922280628724
# se apresenta como a melhor performante rede neural não profunda que conseguimos através da técnica de grid search.